# 🚀 [WBB 와바바] PP-OCRv3 스트리밍 한글 채팅 파인튜닝
- 담당: 김관식 (공동개발자)
- 환경: Google Colab T4 GPU
- 목적: 반투명 채팅창 배경 노이즈 제거 및 한글 채팅 인식률 개선

In [ ]:
# 1. PaddleOCR 공식 레포지토리 클론 및 한국어 폰트 설치
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -r requirements.txt
!pip install text_renderer albumentations
!apt-get install -y fonts-nanum

In [ ]:
# 2. 한국어 PP-OCRv3 Recognition 사전학습 베이스 가중치 다운로드
!wget -nc https://paddleocr.bj.bcebos.com/PP-OCRv3/korean/korean_PP-OCRv3_rec_train.tar
!tar -xvf korean_PP-OCRv3_rec_train.tar
print('✅ 베이스 모델 준비 완료')

In [ ]:
# 3. 업로드된 chat_corpus.txt 단어장 확인
import os
corpus_path = '/content/chat_corpus.txt'
if os.path.exists(corpus_path):
    with open(corpus_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f'✅ 단어장 로드 성공: 총 {len(lines)}개 스트리밍 어휘 확보')
else:
    print('⚠️ 좌측 파일 탐색기(/content/)에 chat_corpus.txt를 먼저 업로드하세요!')

In [ ]:
# 4. PP-OCRv3 Recognition 파인튜닝 실행 (20 Epoch)
!python tools/train.py -c configs/rec/PP-OCRv3/korean_PP-OCRv3_rec.yml \
    -o Global.pretrained_model=korean_PP-OCRv3_rec_train/best_accuracy \
       Global.epoch_num=20 \
       Global.save_model_dir=./output/wbb_custom_ppocr/

In [ ]:
# 5. 배포용 경량 추론 모델(Inference Model) 변환 및 압축
!python tools/export_model.py -c configs/rec/PP-OCRv3/korean_PP-OCRv3_rec.yml \
    -o Global.pretrained_model=./output/wbb_custom_ppocr/best_accuracy \
       Global.save_inference_dir=./wbb_ppocr_inference/

!zip -r wbb_ppocr_inference.zip ./wbb_ppocr_inference
print('🎉 변환 완료! 좌측 파일 목록에서 wbb_ppocr_inference.zip을 다운로드하세요.')